# 23CSE301 Machine Learning — Capstone Project
## Classification Track, Part A

**Dataset:** Comprehensive Diabetes Clinical Dataset — 100,000 records, 16 columns
**Target:** `diabetes` — binary, 0 = non-diabetic, 1 = diabetic
**Task:** supervised binary classification

### Problem statement

Predict whether a patient has diabetes from routinely collected clinical and
demographic measurements: age, BMI, HbA1c, blood glucose, smoking history,
hypertension and heart disease.

### Reproducibility

`random_state = 42` is used everywhere a seed is accepted. Every transformer
that learns a parameter is fitted on the training split only.

---
# 1. Data Loading and Audit

## 1.1 Imports and configuration

One configuration cell fixes the seed, the plotting style and a colourblind-safe
palette so every figure later in the notebook is consistent.

In [1]:
import os
import glob
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# one seed for the whole notebook, so every run reproduces exactly
RANDOM_STATE = 42
TEST_SIZE = 0.20
PALETTE = "colorblind"

np.random.seed(RANDOM_STATE)
warnings.filterwarnings("ignore", category=FutureWarning)

sns.set_theme(style="whitegrid", palette=PALETTE)
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.titlesize": 12,
    "axes.titleweight": "bold",
    "axes.labelsize": 10,
})

# reused whenever a plot is coloured by class
CLASS_COLOURS = {0: sns.color_palette(PALETTE)[0], 1: sns.color_palette(PALETTE)[3]}
CLASS_LABELS = {0: "Non-diabetic (0)", 1: "Diabetic (1)"}

print("pandas      ", pd.__version__)
print("numpy       ", np.__version__)
import sklearn; print("scikit-learn", sklearn.__version__)
print("seaborn     ", sns.__version__)
print("\nRANDOM_STATE =", RANDOM_STATE)

pandas       3.0.6
numpy        2.5.3
scikit-learn 1.9.1
seaborn      0.13.2

RANDOM_STATE = 42


## 1.2 Load the dataset

The loader resolves any `*.csv` in `../data/` rather than hard-coding a filename,
and fails with a clear message if the file is missing.

In [2]:
DATA_DIR = os.path.join("..", "data")

csv_files = sorted(glob.glob(os.path.join(DATA_DIR, "*.csv")))
if not csv_files:
    raise FileNotFoundError(
        f"No CSV found in {os.path.abspath(DATA_DIR)}. "
        "See data/README.md for the dataset description."
    )

DATA_PATH = csv_files[0]
df_raw = pd.read_csv(DATA_PATH)

print("File :", os.path.abspath(DATA_PATH))
print(f"Shape: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")
df_raw.head()

File : /Users/renexaan/Documents/glucotrack/data/diabetes_dataset.csv
Shape: 100,000 rows x 16 columns


,year,gender,age,location,race:AfricanAmerican,race:Asian,race:Caucasian,race:Hispanic,race:Other,hypertension,heart_disease,smoking_history,bmi,hbA1c_level,blood_glucose_level,diabetes
0,2020,Female,32.0,Alabama,0,0,0,0,1,0,0,never,27.32,5.0,100,0
1,2015,Female,29.0,Alabama,0,1,0,0,0,0,0,never,19.95,5.0,90,0
2,2015,Male,18.0,Alabama,0,0,0,0,1,0,0,never,23.76,4.8,160,0
3,2015,Male,41.0,Alabama,0,0,1,0,0,0,0,never,27.32,4.0,159,0
4,2016,Female,52.0,Alabama,1,0,0,0,0,0,0,never,23.75,6.5,90,0


## 1.3 Structural audit

A single consolidated table is easier to read than separate `dtypes`, `isna()` and
`nunique()` calls. It reports, per column: the dtype, how many values are missing,
how many distinct values there are, and a sample of those values.

In [3]:
audit = pd.DataFrame({
    "dtype":     df_raw.dtypes.astype(str),
    "non_null":  df_raw.notna().sum(),
    "missing":   df_raw.isna().sum(),
    "missing_%": (df_raw.isna().mean() * 100).round(3),
    "n_unique":  df_raw.nunique(),
})
# a few example values make it obvious which columns are categorical vs continuous
audit["example_values"] = [
    ", ".join(map(str, df_raw[c].dropna().unique()[:4]))[:55] for c in df_raw.columns
]

print("Total missing values :", int(df_raw.isna().sum().sum()))
print("Duplicate rows       :", int(df_raw.duplicated().sum()))
audit

Total missing values : 0


Duplicate rows       : 14


,dtype,non_null,missing,missing_%,n_unique,example_values
year,int64,100000,0,0.0,7,"2020, 2015, 2016, 2019"
gender,str,100000,0,0.0,3,"Female, Male, Other"
age,float64,100000,0,0.0,102,"32.0, 29.0, 18.0, 41.0"
location,str,100000,0,0.0,55,"Alabama, Alaska, Arizona, Arkansas"
race:AfricanAmerican,int64,100000,0,0.0,2,"0, 1"
race:Asian,int64,100000,0,0.0,2,"0, 1"
race:Caucasian,int64,100000,0,0.0,2,"0, 1"
race:Hispanic,int64,100000,0,0.0,2,"0, 1"
race:Other,int64,100000,0,0.0,2,"1, 0"
hypertension,int64,100000,0,0.0,2,"0, 1"


In [4]:
# distribution of the numeric columns: range, centre and spread
df_raw.describe().T.round(3)

,count,mean,std,min,25%,50%,75%,max
year,100000.0,2018.361,1.345,2015.00,2019.00,2019.00,2019.00,2022.00
age,100000.0,41.886,22.517,0.08,24.00,43.00,60.00,80.00
race:AfricanAmerican,100000.0,0.202,0.402,0.00,0.00,0.00,0.00,1.00
race:Asian,100000.0,0.200,0.400,0.00,0.00,0.00,0.00,1.00
race:Caucasian,100000.0,0.199,0.399,0.00,0.00,0.00,0.00,1.00
race:Hispanic,100000.0,0.199,0.399,0.00,0.00,0.00,0.00,1.00
race:Other,100000.0,0.200,0.400,0.00,0.00,0.00,0.00,1.00
hypertension,100000.0,0.075,0.263,0.00,0.00,0.00,0.00,1.00
heart_disease,100000.0,0.039,0.195,0.00,0.00,0.00,0.00,1.00
bmi,100000.0,27.321,6.637,10.01,23.63,27.32,29.58,95.69


## 1.4 Target variable and class balance

Class balance determines both the metrics that are meaningful and how the
train/test split has to be performed, so it is measured before anything else.

In [5]:
TARGET = "diabetes"

counts = df_raw[TARGET].value_counts().sort_index()
pcts = df_raw[TARGET].value_counts(normalize=True).sort_index() * 100

summary = pd.DataFrame({
    "class":      [CLASS_LABELS[i] for i in counts.index],
    "count":      counts.values,
    "percentage": pcts.round(2).values,
})

imbalance_ratio = counts.max() / counts.min()

print(summary.to_string(index=False))
print(f"\nMajority class  : {pcts.max():.2f}%")
print(f"Minority class  : {pcts.min():.2f}%")
print(f"Imbalance ratio : {imbalance_ratio:.2f} : 1")
print(f"\nA model predicting the majority class for every row would score "
      f"{pcts.max():.1f}% accuracy while identifying no diabetic patients.")

           class  count  percentage
Non-diabetic (0)  91500        91.5
    Diabetic (1)   8500         8.5

Majority class  : 91.50%
Minority class  : 8.50%
Imbalance ratio : 10.76 : 1

A model predicting the majority class for every row would score 91.5% accuracy while identifying no diabetic patients.


### Observation — 1

- 100,000 rows, 16 columns, and no missing values anywhere in the frame.
- 14 exact duplicate rows are present.
- Class split is 91,500 non-diabetic to 8,500 diabetic, a ratio of 10.76 to 1.
- `race` arrives already one-hot encoded across five indicator columns rather than as a single categorical column.
- `smoking_history` includes a "No Info" level, which records missingness rather than a behaviour.
- `hbA1c_level` and `blood_glucose_level` each take only 18 distinct values, and `bmi` has one value repeated far more than any other.